<a href="https://colab.research.google.com/github/AcxelCalle/computational-physics/blob/main/CoiledRope.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Coiled Rope


**Libro**: Marion, Thornton -Classical Dynamics of Particles and Systems

**Problema**: 7.18

<img src="https://github.com/AcxelCalle/computational-physics/blob/main/problem7.18.PNG?raw=true" width="400">


In [1]:
# Importación de las librerías
import sympy as smp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

In [2]:
# Definimos las contantes
m, g, l, t, R = smp.symbols('m g l t R', real=True, pos=True)

# Definimos la coordenada generalizada

alpha = smp.Function(r'\alpha')(t)

# Definimos las posiciones
x = R*smp.sin(alpha) + (l-R*alpha)*smp.sin(smp.pi/2-alpha)
y = R*smp.cos(alpha) - (l-R*alpha)*smp.cos(smp.pi/2-alpha)

In [3]:
# Contruimos el lagrangiano
T = 1/2 * m * (smp.diff(x, t)**2 + smp.diff(y, t)**2)
V = m * g * y
L = T - V

In [4]:
# Definimos las ecuaciones de Euler-Lagrange
dalpha = smp.diff(alpha, t)

EL1 = smp.diff(smp.diff(L, dalpha), t) - smp.diff(L, alpha)

# Definimos d^alpha/dt^2 para resolver la ecuación
ddalpha = smp.diff(alpha, t, 2)
# Resolvemos la ecuacion
alpha_sol = smp.solve(EL1, ddalpha)

In [5]:
alpha_sol[0]


(R*Derivative(\alpha(t), t)**2 + g*cos(\alpha(t)))/(-R*\alpha(t) + l)

In [6]:
alpha_sol_simplificada = smp.cancel(alpha_sol[0])
alpha_sol_simplificada

(R*Derivative(\alpha(t), t)**2 + g*cos(\alpha(t)))/(-R*\alpha(t) + l)

In [7]:
# linealizamos el problema haciendo que z=dalpha/dt
dalpha = smp.diff(alpha, t)
ddalpha_num = smp.lambdify((t, m, g, l, R, dalpha, alpha), alpha_sol_simplificada)
dalpha_num = smp.lambdify(dalpha, dalpha)

In [8]:
# Importamos tqdm para ver el avance del cálculo numérico
from tqdm import tqdm

# Creamos una lista mutable para guardar el tiempo máximo alcanzado
# (Usamos una lista para poder modificarla dentro de la función)
t_max_alcanzado = [0.0]
t_final = 10.0

# Inicializamos la barra de progreso
pbar = tqdm(total=100, desc="Simulando EDO", unit="%")


# Hacemos la martiz dSdt
def dSdt(t, S, m, g, l, R):
    # Actualizamos la barra solo si el solver avanza a un nuevo récord de tiempo
    if t > t_max_alcanzado[0]:
        # Calculamos el incremento en porcentaje
        incremento = ((t - t_max_alcanzado[0]) / t_final) * 100
        pbar.update(incremento)
        t_max_alcanzado[0] = t

    alpha, z = S
    return [
        z,
        ddalpha_num(t, m, g, l, R, z, alpha)
    ]

Simulando EDO:   0%|          | 0/100 [00:00<?, ?%/s]

In [9]:
# importamos el método solve_ivp de scipy
from scipy.integrate import solve_ivp


# Parámetros del sistema
t_num = np.linspace(0, 30, 100000)
m_num = 1
g_num = 9.81
l_num = 5
R_num = 1

# Condiciones iniciales (Puse un ángulo muy pequeño para que empiece a caer si 0 es equilibrio)
alpha0 = 0.001
z0 = 1e-5
S0 = [alpha0, z0]

def cuerdaCompleta(t, S, m, g, l, R):
    alpha = S[0]
    longitud_restante = l - R * alpha

    # Detenemos cuando queda una longitud minúscula (ej: 0.01) para evadir la singularidad
    return longitud_restante - 0.01

cuerdaCompleta.terminal = True
cuerdaCompleta.direction = -1

# Resolvemos la ecuación diferencial (LSODA o BDF son excelentes aquí)
sol = solve_ivp(dSdt, [0, 30], S0, args=(m_num, g_num, l_num, R_num),
                t_eval=t_num, events=cuerdaCompleta, method='BDF')
# Cerramos la barra al terminar
pbar.n = 100
pbar.refresh()
pbar.close()

Simulando EDO: 100%|██████████| 100/100 [00:00<00:00, 478.65%/s] 


In [10]:
sol

  message: A termination event occurred.
  success: True
   status: 1
        t: [ 0.000e+00  3.000e-04 ...  2.067e+00  2.067e+00]
        y: [[ 1.000e-03  1.000e-03 ...  4.931e+00  4.952e+00]
            [ 1.000e-05  5.987e-04 ...  5.780e+01  8.261e+01]]
      sol: None
 t_events: [array([ 2.068e+00])]
 y_events: [array([[ 4.990e+00,  4.062e+02]])]
     nfev: 270
     njev: 10
      nlu: 19

In [11]:
##############################################################
#                 ANIMACIÓN DEL SISTEMA
##############################################################

import matplotlib.animation as animation
from IPython.display import HTML
import matplotlib.pyplot as plt
import numpy as np

matplotlib.rcParams['animation.embed_limit'] = 50.0

dt_fisico = sol.t[1] - sol.t[0]

# 2. Forzamos al navegador a trabajar a unos fluidos ~33 FPS (unos 30 ms por frame)
# Calculamos cuántos cuadros debemos saltarnos para lograr eso
tiempo_por_frame_deseado = 0.030 # 30 milisegundos
salto = max(1, int(tiempo_por_frame_deseado / dt_fisico))

alpha_arr = sol.y[0][::salto]
frames_totales = len(alpha_arr)
# 1. Extraemos el arreglo de tiempo con el mismo salto que usaste para los frames
t_arr = sol.t[::salto]

# 1. Extraemos los ángulos calculados por el solver
# alpha_arr = sol.y[0]
# frames_totales = len(alpha_arr)

# 2. Calculamos las posiciones en cada instante
# (Simplificando sin(pi/2 - a) = cos(a) y cos(pi/2 - a) = sin(a))
x_masa = R_num * np.sin(alpha_arr) + (l_num - R_num * alpha_arr) * np.cos(alpha_arr)
y_masa = R_num * np.cos(alpha_arr) - (l_num - R_num * alpha_arr) * np.sin(alpha_arr)

x_tan = R_num * np.sin(alpha_arr)
y_tan = R_num * np.cos(alpha_arr)

# 3. Configuración del gráfico
fig, ax = plt.subplots(figsize=(7, 7))
ax.set_aspect('equal')
rango = l_num + R_num
ax.set_xlim(-rango, rango)
ax.set_ylim(-rango, rango)
ax.grid(True, linestyle='--', alpha=0.6)
ax.set_title('Masa enrollándose en el cilindro')

# Dibujamos el cilindro central estático
theta_circ = np.linspace(0, 2*np.pi, 100)
ax.plot(R_num * np.cos(theta_circ), R_num * np.sin(theta_circ), 'k-', lw=2)

# Elementos dinámicos
cuerda, = ax.plot([], [], 'k-', lw=1.5)
punto_tan, = ax.plot([], [], 'o', markersize=1.5)
masa, = ax.plot([], [], 'bo', markersize=8)
rastro, = ax.plot([], [], 'b-', alpha=0.3)
tiempo = ax.text(0.05, 0.95, '', transform=ax.transAxes, fontsize=12,
                 bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

hist_x, hist_y = [], []

# 4. Funciones de animación
def init():
    cuerda.set_data([], [])
    punto_tan.set_data([], [])
    masa.set_data([], [])
    rastro.set_data([], [])
    return cuerda, punto_tan, masa, rastro, tiempo

def animate(i):
    px, py = x_masa[i], y_masa[i]
    tx, ty = x_tan[i], y_tan[i]

    # La cuerda va desde el punto de tangencia hasta la masa
    cuerda.set_data([tx, px], [ty, py])
    punto_tan.set_data([tx], [ty])
    masa.set_data([px], [py])

    hist_x.append(px)
    hist_y.append(py)
    rastro.set_data(hist_x, hist_y)

    # 3. Actualizamos el valor del tiempo en cada frame
    tiempo.set_text(f'Tiempo: {t_arr[i]:.3f} s')

    return cuerda, punto_tan, masa, rastro, tiempo

# 4. Calculamos el intervalo EXACTO en milisegundos para dárselo a FuncAnimation
intervalo_real_ms = dt_fisico * salto * 1000

# 5. Generar y mostrar la animación
ani = animation.FuncAnimation(fig, animate, frames=frames_totales,
                              init_func=init, blit=True, interval=intervalo_real_ms)

plt.close()
HTML(ani.to_jshtml())